Looking at how shuffling the dataset changes network performance.

``` python
import json
from landlab_torch_tools import LandlabBatchDataset, GridShuffle, HorizontalSwap
from ThreeLayerCNNRegressor import ThreeLayerCNNRegressor
import torch
import pandas as pd
import matplotlib.pyplot as plt
```

``` python
stats_path = '../../data/model_stats.json'
db_path = '../../data/model_runs.db'
dataset_path = '../../data/0/elevation'
label_query = 'SELECT "model_param.diffuser.D" / "model_param.streampower.k"  FROM model_run_params'
swap_results_path = "../../analysis/swap_results.csv"
shuff_results_path = "../../analysis/shuffle_results.csv"
weight_path = "../../weights/n0_model_dems_0_DoK_weights.pt"
batch_size = 64
```

``` python
with open(stats_path) as f:
    stats = json.load(f)
label_stats = stats['DoK']
shuffled_dataset = LandlabBatchDataset(
    db_path = db_path,
    dataset_dir = dataset_path,
    label_query = label_query,
    transform = HorizontalSwap(),
    **stats['0']['elevation'],
    **stats['DoK']
)
shuffled_dataloader = torch.utils.data.DataLoader(
    shuffled_dataset,
    batch_size = batch_size
)
model = ThreeLayerCNNRegressor()
model.load_state_dict(torch.load(weight_path))
```

``` python
predictions = []
true_labels = []
for data, labels in shuffled_dataloader:
    outputs = model(data)
    predictions += outputs
    true_labels += labels

predictions = [float(p)*label_stats['labels_std'] + label_stats['labels_mean'] for p in predictions]
true_labels = [float(l)*label_stats['labels_std'] + label_stats['labels_mean'] for l in true_labels]

df = pd.DataFrame({'predictions': predictions,
              'true_labels': true_labels})
df.to_csv(swap_results_path)
```

``` python
plt.scatter(true_labels, predictions)
```

``` python
shuffled_dataset = LandlabBatchDataset(
    db_path = db_path,
    dataset_dir = dataset_path,
    label_query = label_query,
    transform = GridShuffle(),
    **stats['0']['elevation'],
    **stats['DoK']
)
shuffled_dataloader = torch.utils.data.DataLoader(
    shuffled_dataset,
    batch_size = batch_size
)
model = ThreeLayerCNNRegressor()
model.load_state_dict(torch.load(weight_path))
```

``` python
predictions = []
true_labels = []
for data, labels in shuffled_dataloader:
    outputs = model(data)
    predictions += outputs
    true_labels += labels

predictions = [float(p)*label_stats['labels_std'] + label_stats['labels_mean'] for p in predictions]
true_labels = [float(l)*label_stats['labels_std'] + label_stats['labels_mean'] for l in true_labels]

df = pd.DataFrame({'predictions': predictions,
              'true_labels': true_labels})
df.to_csv(shuff_results_path)
```

``` python
plt.scatter(true_labels, predictions)
plt.show()
```